In [ ]:
import os
import json
import torch
import cv2
import numpy as np
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import albumentations as A
from torch.utils.data import Dataset
from transformers import (
    Mask2FormerImageProcessor,
    Mask2FormerForUniversalSegmentation,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback
)

# 1. Augmentations (Horizontal flips & subtle spatial distortion)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(translate_percent=0.05, scale=(0.95, 1.05), rotate=0, p=0.5),
    A.ElasticTransform(p=0.2, alpha=1, sigma=50),
])

# 2. COCO Dataset Loader (Parses COCO JSON directly)
class COCOWireDataset(Dataset):
    def __init__(self, json_path, image_dir, processor, transforms=None):
        self.image_dir = image_dir
        self.processor = processor
        self.transforms = transforms

        # Load COCO JSON
        with open(json_path, 'r') as f:
            self.coco = json.load(f)

        # Build category mapping (e.g., 'anode' -> 0, 'cathode' -> 1)
        self.categories = self.coco.get('categories', [])
        self.cat_id_to_sem_id = {cat['id']: idx for idx, cat in enumerate(self.categories)}
        
        # Map images to their annotations
        self.images = self.coco.get('images', [])
        self.img_to_anns = {}
        for ann in self.coco.get('annotations', []):
            self.img_to_anns.setdefault(ann['image_id'], []).append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info['id']
        file_name = img_info['file_name']
        image_path = os.path.join(self.image_dir, file_name)

        # Load image
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Image not found: {image_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        # Generate 2D Instance Mask (Background is 255)
        anns = self.img_to_anns.get(img_id, [])
        instance_mask = np.full((h, w), 255, dtype=np.int32)
        instance_id_to_semantic_id = {}

        for inst_idx, ann in enumerate(anns, start=1):
            # ---------------------------------------------------------
            # NEW: Draw precise Polygons instead of Bounding Boxes
            # ---------------------------------------------------------
            if 'segmentation' in ann and ann['segmentation']:
                for poly in ann['segmentation']:
                    # Convert [x1, y1, x2, y2...] to numpy array of shape (N, 2)
                    pts = np.array(poly, np.int32).reshape((-1, 2))
                    # Paint the exact polygon shape onto the mask with the instance ID
                    cv2.fillPoly(instance_mask, [pts], inst_idx)
            else:
                # Fallback to bounding box ONLY if polygon data is missing
                bbox = ann['bbox'] 
                x, y, bw, bh = [int(v) for v in bbox]
                x1, y1 = max(0, x), max(0, y)
                x2, y2 = min(w, x + bw), min(h, y + bh)
                instance_mask[y1:y2, x1:x2] = inst_idx
            # ---------------------------------------------------------

            # Map category to 0 (anode) or 1 (cathode)
            cat_id = ann['category_id']
            instance_id_to_semantic_id[inst_idx] = self.cat_id_to_sem_id[cat_id]

        # Apply Albumentations
        # Apply Albumentations
        if self.transforms:
            augmented = self.transforms(image=image, mask=instance_mask)
            image = augmented['image']
            instance_mask = augmented['mask']
            
            # FIX: Albumentations pads empty borders with 0. Change them back to background (255)
            instance_mask[instance_mask == 0] = 255

        # Build ground-truth semantic map for validation metrics
        semantic_target = np.zeros((h, w), dtype=np.uint8)
        for inst_id, sem_id in instance_id_to_semantic_id.items():
            semantic_target[instance_mask == inst_id] = sem_id + 1

        return image, instance_mask, instance_id_to_semantic_id, semantic_target


def collate_fn(batch):
    images = [item[0] for item in batch]
    segmentation_maps = [item[1] for item in batch]
    instance_id_to_semantic_id = [item[2] for item in batch]

    inputs = processor(
        images=images,
        segmentation_maps=segmentation_maps,
        instance_id_to_semantic_id=instance_id_to_semantic_id,
        return_tensors="pt",
    )
    return inputs


# 3. Custom Evaluation Callback (For Epoch Metric Logging)
class ThresholdEvaluationCallback(TrainerCallback):
    def __init__(self, val_dataset, processor, anode_thresh=0.50, cathode_thresh=0.50):
        self.val_dataset = val_dataset
        self.processor = processor
        self.anode_thresh = anode_thresh
        self.cathode_thresh = cathode_thresh

    def on_evaluate(self, args, state, control, model, **kwargs):
        model.eval()
        device = args.device

        ious_anode, dices_anode = [], []
        ious_cathode, dices_cathode = [], []

        for idx in range(len(self.val_dataset)):
            image, _, _, semantic_target = self.val_dataset[idx]
            inputs = self.processor(images=[image], return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = model(**inputs)
                class_logits = outputs.class_queries_logits
                mask_logits = outputs.masks_queries_logits

                class_probs = class_logits.softmax(dim=-1)[..., :-1]
                mask_probs = mask_logits.sigmoid()

                # Combine query predictions into class spatial probabilities
                semantic_probs = torch.einsum("bqc, bqhw -> bchw", class_probs, mask_probs)

                # FIX: Resize model's 96x96 output back to the 540x540 image size
                semantic_probs = torch.nn.functional.interpolate(
                    semantic_probs, 
                    size=semantic_target.shape, 
                    mode="bilinear", 
                    align_corners=False
                )

                pred_anode = (semantic_probs[0, 0] > self.anode_thresh).cpu().numpy()
                pred_cathode = (semantic_probs[0, 1] > self.cathode_thresh).cpu().numpy()

            target_anode = (semantic_target == 1)
            target_cathode = (semantic_target == 2)

            def calc_metrics(pred, target):
                inter = np.logical_and(pred, target).sum()
                union = np.logical_or(pred, target).sum()
                iou = inter / union if union > 0 else np.nan
                dice = 2 * inter / (pred.sum() + target.sum()) if (pred.sum() + target.sum()) > 0 else np.nan
                return iou, dice

            iou_a, dice_a = calc_metrics(pred_anode, target_anode)
            iou_c, dice_c = calc_metrics(pred_cathode, target_cathode)

            if not np.isnan(iou_a): ious_anode.append(iou_a)
            if not np.isnan(dice_a): dices_anode.append(dice_a)
            if not np.isnan(iou_c): ious_cathode.append(iou_c)
            if not np.isnan(dice_c): dices_cathode.append(dice_c)

        print(f"\n--- Epoch {state.epoch} Validation Metrics ---")
        print(f"Anode   (Thresh {self.anode_thresh:.2f}) | IoU: {np.mean(ious_anode):.4f} | Dice: {np.mean(dices_anode):.4f}")
        print(f"Cathode (Thresh {self.cathode_thresh:.2f}) | IoU: {np.mean(ious_cathode):.4f} | Dice: {np.mean(dices_cathode):.4f}")
        print("-------------------------------------------\n")


if __name__ == "__main__":
    # --- Paths for your Cropped Dataset ---
    BASE_DIR = r"E:\Prithu\25 sep\cropped"
    JSON_PATH = os.path.join(BASE_DIR, "annotations.json")
    IMAGE_DIR = os.path.join(BASE_DIR, "images")
    
    MODEL_CHECKPOINT = "facebook/mask2former-swin-base-coco-instance"
    ID2LABEL = {0: "anode", 1: "cathode"}

    processor = Mask2FormerImageProcessor.from_pretrained(
        MODEL_CHECKPOINT, 
        ignore_index=255, 
        reduce_labels=False 
    )
    model = Mask2FormerForUniversalSegmentation.from_pretrained(
        MODEL_CHECKPOINT, id2label=ID2LABEL, ignore_mismatched_sizes=True
    )

    # Initialize Dataset
    full_dataset = COCOWireDataset(
        json_path=JSON_PATH, 
        image_dir=IMAGE_DIR, 
        processor=processor, 
        transforms=train_transform
    )

    # Split dataset into Train (80%) and Validation (20%)
    val_size = int(0.2 * len(full_dataset))
    train_size = len(full_dataset) - val_size
    # Fix random seed for consistent train/val splits across runs
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size], generator=generator)

    # Optimizer: Layer-wise Learning Rate Decay
    backbone_params = [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad]
    decoder_params = [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]
    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": 1e-5},
        {"params": decoder_params, "lr": 1e-4},
    ], weight_decay=0.05)

    # Training Configuration
    training_args = TrainingArguments(
        output_dir=os.path.join(BASE_DIR, "mask2former_model_final"),
        num_train_epochs=400,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=3,
        per_device_train_batch_size=24,
        per_device_eval_batch_size=24,
        dataloader_num_workers=0,
        fp16=True,
        remove_unused_columns=False,
    )

    early_stop = EarlyStoppingCallback(early_stopping_patience=50)
    custom_metrics = ThresholdEvaluationCallback(
        val_dataset=val_dataset,
        processor=processor,
        anode_thresh=0.50,
        cathode_thresh=0.50
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collate_fn,
        optimizers=(optimizer, None),
        callbacks=[early_stop, custom_metrics]
    )

    print(f"Loaded {len(full_dataset)} images from {BASE_DIR}")
    print("Starting Training...")
    trainer.train()

In [ ]:
import os
import cv2
import torch
import numpy as np
import math
import matplotlib.pyplot as plt

from skimage.morphology import skeletonize
from skimage.graph import route_through_array
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

# =========================================================
# CONFIG
# =========================================================

BASE_DIR = r"E:\Prithu\25 sep\cropped"
MODEL_DIR = os.path.join(BASE_DIR, "mask2former_model_final")
INPUT_DIR = r"E:\Prithu\Anode\New folder"
OUTPUT_DIR = os.path.join(BASE_DIR, "predictions_hybrid_skimage")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CROP_SIZE = 540
OVERLAP = 150
STRIDE = CROP_SIZE - OVERLAP
SCORE_THRESHOLD = 0.49
MICRONS_PER_PIXEL = 1.7943

CLASS_MAPPING = {
    1: {"name": "Anode Wire", "color": (255, 0, 0), "alpha": 0.6, "order": 1},      # Red
    3: {"name": "Cathode Wire", "color": (0, 0, 255), "alpha": 0.6, "order": 2},    # Blue
}

# =========================================================
# HYBRID WHITE PIXEL & APEX LOGIC
# =========================================================

def get_hybrid_white_pixels(rgb_image):
    gray = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    
    _, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 41, -4)
    
    binary = (otsu > 0) & (adaptive > 0)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary = cv2.morphologyEx(binary.astype(np.uint8)*255, cv2.MORPH_CLOSE, kernel) > 0
    return binary

def find_physical_apex(cx, cy, white_pixels, comp_mask, H, W, search_radius=20):
    y_min = max(0, cy - search_radius)
    y_max = min(H - 1, cy + 10)
    x_min = max(0, cx - search_radius)
    x_max = min(W - 1, cx + search_radius)
    
    roi_white = white_pixels[y_min:y_max+1, x_min:x_max+1]
    roi_mask = comp_mask[y_min:y_max+1, x_min:x_max+1]
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    roi_mask_dil = cv2.dilate(roi_mask.astype(np.uint8), kernel) > 0
    
    valid_white = roi_white & roi_mask_dil
    v_ys, v_xs = np.where(valid_white)
    
    if len(v_ys) > 0:
        abs_min_y = np.min(v_ys)
        best_xs = v_xs[v_ys == abs_min_y]
        return int(np.median(best_xs)) + x_min, int(abs_min_y) + y_min
    
    return cx, cy

# =========================================================
# 90-DEGREE SCANNER (MASK-BASED FOR 100% RELIABILITY)
# =========================================================

def find_perfect_horizontal_intersection(cx, cy, mask_anode_bool, direction, W, H, max_dy=30):
    """
    Scans the segmentation mask (not the skeleton) to guarantee we find the wire even if 
    it is dark at the bottom. Returns the strict median X while locking Y to perfectly 180 degrees.
    """
    scan_range = range(cx, W) if direction == 1 else range(cx, -1, -1)
    
    def get_mid_x(y):
        hits = []
        in_wire = False
        for x in scan_range:
            if mask_anode_bool[y, x]:
                hits.append(x)
                in_wire = True
            elif in_wire:
                break # Reached the other side of the FIRST wire hit
        return int(np.median(hits)) if hits else None

    # Check exact line first (180 degree lock)
    mid = get_mid_x(cy)
    if mid is not None:
        return (mid, cy)
        
    # Vertical tolerance if the mask has a tiny gap, but still forced to exact 'cy'
    for dy in range(1, max_dy + 1):
        if cy - dy >= 0:
            mid = get_mid_x(cy - dy)
            if mid is not None: return (mid, cy)
        if cy + dy < H:
            mid = get_mid_x(cy + dy)
            if mid is not None: return (mid, cy)
            
    return None

# =========================================================
# MODEL LOGIC
# =========================================================

def load_model(model_dir, device):
    checkpoint_folders = [f for f in os.listdir(model_dir) if f.startswith("checkpoint-")]
    if checkpoint_folders:
        checkpoint_folders.sort(key=lambda x: int(x.split("-")[-1]))
        latest_ckpt = os.path.join(model_dir, checkpoint_folders[-1])
    else:
        latest_ckpt = model_dir 
        
    print(f"Loading model from: {latest_ckpt}")
    processor = Mask2FormerImageProcessor.from_pretrained(
        "facebook/mask2former-swin-base-coco-instance", ignore_index=255, reduce_labels=False
    )
    model = Mask2FormerForUniversalSegmentation.from_pretrained(latest_ckpt)
    model.to(device)
    model.eval()
    return processor, model

def clean_masks(masks):
    overlap_wires = masks[3] & masks[1]
    masks[3][overlap_wires] = False
    masks[1][overlap_wires] = False
    return masks

# =========================================================
# DISTANCE & TRACING INTEGRATION
# =========================================================

def compute_distance_and_lines(masks, rgb_image): 
    mask_anode_wire = masks[1].astype(np.uint8) * 255     
    mask_cath_wire = masks[3].astype(np.uint8) * 255      
    img_height, img_width = mask_anode_wire.shape

    mask_anode_bool = mask_anode_wire > 0

    # 1. Physical White Pixels
    white_pixels = get_hybrid_white_pixels(rgb_image)

    # 2. Hybrid Skeleton Generation (White pixels inside segmentation)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    dilated_seg = cv2.dilate(mask_anode_wire, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))) > 0
    
    hybrid_core = white_pixels & dilated_seg
    hybrid_core = cv2.morphologyEx(hybrid_core.astype(np.uint8), cv2.MORPH_CLOSE, kernel_close) > 0
    skel_anode = skeletonize(hybrid_core)
    
    num_a, labels_a, stats_a, _ = cv2.connectedComponentsWithStats(mask_anode_wire)

    distances_px = []
    traced_paths = [] 

    # STEP 1: Cathode Tops
    num_c, labels_c, stats_c, _ = cv2.connectedComponentsWithStats(mask_cath_wire)
    cathodes = []

    for i in range(1, num_c):
        if stats_c[i, cv2.CC_STAT_AREA] < 1000: continue
            
        local_ys, local_xs = np.where(labels_c == i)
        min_y = np.min(local_ys)
        cx = int(np.median(local_xs[local_ys == min_y]))
        cy = int(min_y)
        
        comp_mask = (labels_c == i)
        cx, cy = find_physical_apex(cx, cy, white_pixels, comp_mask, img_height, img_width)
        cathodes.append({'x': cx, 'y': cy})
        
    cathodes.sort(key=lambda c: c['x'])

    for i, cat in enumerate(cathodes):
        cx, cy = cat['x'], cat['y']
        
        # Leftmost cathode traces both sides; others trace right side only.
        directions = [-1, 1] if i == 0 else [1]

        for d in directions:
            # Pass mask_anode_bool to guarantee hit detection
            intersec = find_perfect_horizontal_intersection(cx, cy, mask_anode_bool, d, img_width, img_height)
            if intersec is None: continue
            ix, iy = intersec

            # Identify target Anode
            anode_id = labels_a[iy, ix]
            if anode_id == 0:
                sub = labels_a[max(0, iy-3):min(img_height, iy+4), max(0, ix-3):min(img_width, ix+4)]
                valid_ids = sub[sub > 0]
                if len(valid_ids) > 0: anode_id = valid_ids[0]
            if anode_id == 0: continue
                
            anode_comp_mask = (labels_a == anode_id)

            # Find Anode True Apex
            a_ys, a_xs = np.where(anode_comp_mask)
            min_a_y = np.min(a_ys)
            seed_a_x = int(np.median(a_xs[a_ys == min_a_y]))
            true_apex = find_physical_apex(seed_a_x, min_a_y, white_pixels, anode_comp_mask, img_height, img_width)

            # ========================================================
            # SKIMAGE LIBRARY ROUTING
            # ========================================================
            # Safely expand bounding box to guarantee inclusion of intersection and apex
            min_y_bound = min(np.min(a_ys), iy, true_apex[1])
            max_y_bound = max(np.max(a_ys), iy, true_apex[1])
            min_x_bound = min(np.min(a_xs), ix, true_apex[0])
            max_x_bound = max(np.max(a_xs), ix, true_apex[0])
            
            y0 = max(0, min_y_bound - 30)
            y1 = min(img_height - 1, max_y_bound + 30)
            x0 = max(0, min_x_bound - 30)
            x1 = min(img_width - 1, max_x_bound + 30)

            local_skel = skel_anode[y0:y1+1, x0:x1+1]
            local_mask = anode_comp_mask[y0:y1+1, x0:x1+1]
            
            # Cost Array: Skeleton = 1, Mask = 100, Empty Space = 1000
            local_cost = np.full(local_skel.shape, 1000.0, dtype=np.float32)
            local_cost[local_mask] = 100.0  
            local_cost[local_skel] = 1.0    

            start_rc = (iy - y0, ix - x0)
            end_rc = (true_apex[1] - y0, true_apex[0] - x0)

            try:
                local_path_rc, _ = route_through_array(local_cost, start_rc, end_rc, fully_connected=True)
                raw_path = [(c + x0, r + y0) for r, c in local_path_rc]
            except Exception:
                raw_path = [intersec, true_apex]

            # RDP Smoothing
            pts_array = np.array(raw_path, dtype=np.float32).reshape(-1, 1, 2)
            smoothed_pts = cv2.approxPolyDP(pts_array, epsilon=2.0, closed=False)
            smoothed_path = [(int(p[0][0]), int(p[0][1])) for p in smoothed_pts]
            
            smoothed_path[0] = intersec
            smoothed_path[-1] = true_apex

            # ========================================================
            # DISTANCE CALCULATION (HORIZONTAL JUMP EXCLUDED)
            # ========================================================
            path_distance = 0.0
            for pt_idx in range(len(smoothed_path) - 1):
                x1, y1 = smoothed_path[pt_idx]
                x2, y2 = smoothed_path[pt_idx + 1]
                path_distance += math.hypot(x2 - x1, y2 - y1)

            individual_um = path_distance * MICRONS_PER_PIXEL 
            distances_px.append(path_distance)
            
            traced_paths.append({
                "cath_start": (cx, cy),
                "intersec": intersec,
                "path": smoothed_path,
                "apex": true_apex,
                "dist_um": individual_um
            })

    if len(distances_px) == 0:
        return None, None

    avg_px = np.mean(distances_px)
    avg_um = avg_px * MICRONS_PER_PIXEL  
    return (avg_px, avg_um), traced_paths

# =========================================================
# MAIN
# =========================================================

def main():
    os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print("Initializing Model...")
    processor, model = load_model(MODEL_DIR, device)

    valid_ext = ('.tif', '.tiff', '.png', '.jpg', '.jpeg')
    image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_ext)]
    
    if not image_files: return

    for img_name in image_files:
        print(f"\nProcessing Image: {img_name}")
        img_path = os.path.join(INPUT_DIR, img_name)
        img = cv2.imread(img_path)
        if img is None: continue
            
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        global_masks = { i: np.zeros((H, W), dtype=bool) for i in CLASS_MAPPING }

        with torch.no_grad():
            for y in range(0, H, STRIDE):
                for x in range(0, W, STRIDE):
                    y2, x2 = min(y + CROP_SIZE, H), min(x + CROP_SIZE, W)
                    crop = img[y:y2, x:x2]
                    ch, cw = crop.shape[:2]

                    padded = np.zeros((CROP_SIZE, CROP_SIZE, 3), dtype=np.uint8)
                    padded[:ch, :cw] = crop

                    inputs = processor(images=[padded], return_tensors="pt").to(device)
                    outputs = model(**inputs)

                    class_probs = outputs.class_queries_logits.softmax(dim=-1)[..., :-1]
                    mask_probs = outputs.masks_queries_logits.sigmoid()

                    semantic_probs = torch.einsum("bqc, bqhw -> bchw", class_probs, mask_probs)[0]
                    semantic_probs = torch.nn.functional.interpolate(
                        semantic_probs.unsqueeze(0), size=(CROP_SIZE, CROP_SIZE), mode="bilinear", align_corners=False
                    )[0]

                    global_masks[1][y:y2, x:x2] |= (semantic_probs[0] > SCORE_THRESHOLD).cpu().numpy()[:ch, :cw]
                    global_masks[3][y:y2, x:x2] |= (semantic_probs[1] > SCORE_THRESHOLD).cpu().numpy()[:ch, :cw]

        vis = img.copy()
        for cid, info in sorted(CLASS_MAPPING.items(), key=lambda x: x[1]["order"]):
            m = global_masks[cid]
            if np.any(m):
                overlay = vis.copy()
                overlay[m] = info["color"]
                vis = cv2.addWeighted(overlay, info["alpha"], vis, 1 - info["alpha"], 0)

        cleaned = clean_masks(global_masks)
        result, traced_paths = compute_distance_and_lines(cleaned, img)

        if result is not None:
            for item in traced_paths:
                c_pt, inter_pt, path_pts, apex, d_um = item["cath_start"], item["intersec"], item["path"], item["apex"], item["dist_um"]

                # Yellow PERFECTLY HORIZONTAL 180-degree jump
                cv2.line(vis, c_pt, inter_pt, (0, 255, 255), 1, cv2.LINE_AA)
                
                # White Distance Line
                trace_pts = np.array(path_pts, np.int32).reshape((-1, 1, 2))
                cv2.polylines(vis, [trace_pts], False, (255, 255, 255), 1, cv2.LINE_AA)

                for pt in [c_pt, apex]:
                    top_overlay = vis.copy()
                    cv2.circle(top_overlay, pt, 5, (255, 150, 220), -1, cv2.LINE_AA)
                    vis = cv2.addWeighted(top_overlay, 0.40, vis, 0.60, 0)
                
                cv2.putText(vis, f"{d_um:.1f}um", (apex[0] + 15, apex[1] + 15),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

        out_path = os.path.join(OUTPUT_DIR, f"merged_{img_name}")
        cv2.imwrite(out_path, cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()

In [ ]:

from pathlib import Path
import json

import numpy as np
import onnx
import onnxruntime as ort
import torch
from transformers import (
    Mask2FormerImageProcessor,
    Mask2FormerForUniversalSegmentation,
)

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------

BASE_DIR = Path(r"Y:\Institute\IMFAA\Daten\08_Transfer\Goti_Jenil\Anode final")
MODEL_DIR = BASE_DIR / "mask2former_model_final"
ONNX_DIR = BASE_DIR / "onnx_export"

ONNX_PATH = ONNX_DIR / "mask2former_model.onnx"
METADATA_PATH = ONNX_DIR / "metadata.json"

MODEL_DIR = MODEL_DIR.resolve()
ONNX_DIR.mkdir(parents=True, exist_ok=True)

# Match the processor used by the original inference script.
PROCESSOR_NAME = "facebook/mask2former-swin-base-coco-instance"
CROP_SIZE = 540
OPSET_VERSION = 17


def find_checkpoint(model_dir: Path) -> Path:
    """Find the numerically latest checkpoint, or use model_dir."""
    checkpoints = []

    for path in model_dir.glob("checkpoint-*"):
        if path.is_dir():
            suffix = path.name.removeprefix("checkpoint-")
            if suffix.isdigit():
                checkpoints.append((int(suffix), path))

    if checkpoints:
        return max(checkpoints, key=lambda item: item[0])[1]

    return model_dir


class Mask2FormerONNXWrapper(torch.nn.Module):
    """Expose tensor outputs instead of the Transformers output object."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values, pixel_mask):
        outputs = self.model(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask,
            return_dict=True,
        )

        return (
            outputs.class_queries_logits,
            outputs.masks_queries_logits,
        )


def main():
    checkpoint = find_checkpoint(MODEL_DIR)

    print(f"Loading checkpoint: {checkpoint}")
    print(f"Export destination: {ONNX_PATH}")

    processor = Mask2FormerImageProcessor.from_pretrained(
        PROCESSOR_NAME,
        ignore_index=255,
        reduce_labels=False,
    )

    model = Mask2FormerForUniversalSegmentation.from_pretrained(
        str(checkpoint)
    )
    model.eval()
    model.cpu()

    # Use exactly the same preprocessing convention as the
    # original script: RGB crop -> Hugging Face processor.
    dummy_rgb = np.zeros(
        (CROP_SIZE, CROP_SIZE, 3),
        dtype=np.uint8,
    )

    processed = processor(
        images=[dummy_rgb],
        return_tensors="pt",
    )

    pixel_values = processed["pixel_values"].cpu()

    # The processor normally supplies a pixel mask. Create one
    # only if this Transformers version does not return it.
    pixel_mask = processed.get("pixel_mask")

    if pixel_mask is None:
        pixel_mask = torch.ones(
            (
                pixel_values.shape[0],
                pixel_values.shape[2],
                pixel_values.shape[3],
            ),
            dtype=torch.long,
        )

    pixel_mask = pixel_mask.cpu()

    wrapper = Mask2FormerONNXWrapper(model).eval()

    # Validate the wrapper before exporting.
    with torch.inference_mode():
        expected_class, expected_masks = wrapper(
            pixel_values,
            pixel_mask,
        )

    print("Input tensor:", tuple(pixel_values.shape))
    print("Pixel mask:", tuple(pixel_mask.shape))
    print("Class logits:", tuple(expected_class.shape))
    print("Mask logits:", tuple(expected_masks.shape))

    torch.onnx.export(
        wrapper,
        (pixel_values, pixel_mask),
        str(ONNX_PATH),
        input_names=["pixel_values", "pixel_mask"],
        output_names=[
            "class_queries_logits",
            "masks_queries_logits",
        ],
        dynamic_axes={
            "pixel_values": {0: "batch"},
            "pixel_mask": {0: "batch"},
            "class_queries_logits": {0: "batch"},
            "masks_queries_logits": {0: "batch"},
        },
        opset_version=OPSET_VERSION,
        do_constant_folding=True,
        dynamo=False,
    )

    # Validate ONNX graph structure.
    onnx_model = onnx.load(str(ONNX_PATH))
    onnx.checker.check_model(onnx_model)
    print("ONNX graph validation passed.")

    # Validate that ONNX Runtime can execute the exported model.
    session = ort.InferenceSession(
        str(ONNX_PATH),
        providers=["CPUExecutionProvider"],
    )

    ort_outputs = session.run(
        ["class_queries_logits", "masks_queries_logits"],
        {
            "pixel_values": pixel_values.numpy(),
            "pixel_mask": pixel_mask.numpy().astype(np.int64),
        },
    )

    for name, expected, actual in zip(
        ["class logits", "mask logits"],
        [expected_class, expected_masks],
        ort_outputs,
    ):
        actual = np.asarray(actual)

        if not np.isfinite(actual).all():
            raise RuntimeError(f"Non-finite values in ONNX {name}")

        np.testing.assert_allclose(
            expected.detach().numpy(),
            actual,
            rtol=1e-3,
            atol=1e-4,
            err_msg=f"ONNX mismatch in {name}",
        )

        print(f"{name}: PyTorch/ONNX comparison passed.")

    # Preserve processor configuration for deployment.
    processor.save_pretrained(str(ONNX_DIR))

        # Save model and inference configuration.
    metadata = {
        "model_name": "Mask2Former",
        "checkpoint": str(checkpoint),
        "processor": PROCESSOR_NAME,

        # Sliding-window inference configuration
        "crop_size": CROP_SIZE,
        "overlap": OVERLAP,
        "stride": STRIDE,

        # Segmentation and distance configuration
        "score_threshold": SCORE_THRESHOLD,
        "microns_per_pixel": MICRONS_PER_PIXEL,

        # ONNX configuration
        "onnx_opset": OPSET_VERSION,
        "onnx_file": ONNX_PATH.name,

        # Model input/output names
        "input_names": [
            "pixel_values",
            "pixel_mask",
        ],
        "output_names": [
            "class_queries_logits",
            "masks_queries_logits",
        ],

        # Class IDs used by your original inference code
        "class_mapping": {
            "1": "Anode Wire",
            "3": "Cathode Wire",
        },
    }

    METADATA_PATH.write_text(
        json.dumps(metadata, indent=4),
        encoding="utf-8",
    )

    print(f"Metadata saved: {METADATA_PATH}")

    METADATA_PATH.write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )

    print("\nExport completed successfully.")
    print(f"ONNX model: {ONNX_PATH}")
    print(f"Processor and metadata: {ONNX_DIR}")



main()


Loading checkpoint: \\fssnas2.htw-aalen.de\IMFAA-Daten\08_Transfer\Goti_Jenil\Anode final\mask2former_model_final\checkpoint-22816
Export destination: Y:\Institute\IMFAA\Daten\08_Transfer\Goti_Jenil\Anode final\onnx_export\mask2former_model.onnx


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Input tensor: (1, 3, 384, 384)
Pixel mask: (1, 384, 384)
Class logits: (1, 100, 3)
Mask logits: (1, 100, 96, 96)


C:\Users\3011778\AppData\Local\Temp\ipykernel_20456\3758672671.py:131: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
c:\Users\3011778\b-segmentation\.venv\Lib\site-packages\transformers\models\swin\modeling_swin.py:269: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if width % self.patch_size[1] != 0:
c:\Users\3011778\b-segmentation\.venv\Lib\site-packages\transformers\models\swin\modeling_swin.py:272: TracerWarning: Conv

ONNX graph validation passed.


InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Invalid input name: pixel_mask